树其实是一种特殊的图。但树有很强的限制，比如：
- 只有一个根
- 不能有环
- 父子关系比较明确
- 节点之间不是随便连的
而图更自由，它关注的是：**“谁和谁之间有关系？”**
比如：
- 城市 A 和城市 B 有路相连
- 课程 A 是课程 B 的先修课
- 用户 A 关注了用户 B
- 网页 A 链接到网页 B
所以，图本质上是在描述：对象之间的连接关系。
## 组成
### 顶点（Vertex）
顶点也叫 节点（Node），它是图中的基本单位。
每个顶点通常有：
- 一个标识名（key / id）
- 可能还有额外信息（payload）

> 可以把顶点简单理解为 **图中被研究的对象本身**。
### 边（Edge）
边也叫 弧（Arc），表示两个顶点之间存在某种关系。

### 有向图与无向图
> 在离散数学中都很清楚，这里就不再赘述

### 权重（Weight）
本质上是在描述 **从一个顶点到另一个顶点，需要付出的代价**。
### 路径（Path）与环（Cycle）
路径就是从一个顶点出发，沿着边一步一步走，形成的一串顶点序列。
路径长度有两种理解方式：
- 无权路径长度：只数边的条数。
  - `0 -> 5 -> 2 -> 3`共有 3 条边，所以长度是 3
- 带权路径长度：把路径上所有边的权重加起来。
  - `0 -> 5 权重 2 5 -> 2 权重 1 2 -> 3 权重 9`那么总长度就是12

环就是一条路径从某个顶点出发，最后又回到了这个顶点。

### 无环图（Acyclic Graph）
如果一个图里没有任何环，就叫无环图
如果是有向图且无环，就叫 **有向无环图（DAG, Directed Acyclic Graph）**
DAG 在算法里特别重要，因为很多“依赖关系问题”都可以建模成 DAG：
- 课程先修关系
- 任务调度
- 编译依赖
- 工作流执行顺序
因为如果有环，就会出现逻辑矛盾：
- A 依赖 B
- B 依赖 C
- C 又依赖 A

这就没法开始了。所以 DAG 很适合表示“必须按顺序完成”的问题。


## 抽象数据类型（Graph ADT）
| 序号 | 操作功能 | Python 语法 / 方法 | 详细说明 |
| ---- | -------- | ----------------- | -------- |
| 1 | 创建图 | `Graph()` | 初始化，创建一个空的图结构 |
| 2 | 添加顶点 | `addVertex(vert)` | 向图中插入单个顶点 |
| 3 | 添加无权重有向边 | `addEdge(fromVert, toVert)` | 添加一条从 `fromVert` 指向 `toVert` 的有向边 |
| 4 | 添加带权重有向边 | `addEdge(fromVert, toVert, weight)` | 添加一条带权重的、从 `fromVert` 指向 `toVert` 的有向边 |
| 5 | 获取单个顶点 | `getVertex(vertKey)` | 根据顶点名称/编号，查询并返回对应顶点 |
| 6 | 获取全部顶点 | `getVertices()` | 返回图中所有顶点的集合/列表 |
| 7 | 判断顶点是否存在 | `vert in graph` | 布尔判断：顶点存在返回 `True`，不存在返回 `False` |

### 存储方式
#### 邻接矩阵（Adjacency Matrix）
假设图中有 n 个顶点，就建立一个 n × n 的矩阵。矩阵中：
- 行表示起点
- 列表示终点
如果从 v 到 w 有边，就在 matrix[v][w] 里存值：
- 无权图
    - 有边：存 1
    - 没边：存 0
- 带权图
    - 有边：存权重
    - 没边：存 0 / 无穷 / 特殊值

| 分类 | 核心要点 | 详细说明 |
| :--- | :--- | :--- |
| **优点** | 实现简单 | 底层本质是二维数组，数据结构直观、代码易编写理解 |
|  | 判断两点连通极快 | 查询顶点v、w是否有边，直接读取 `matrix[v][w]`；<br>时间复杂度：$O(1)$ |
| **缺点** | 空间开销大、浪费严重 | 顶点数量为 $|V|$ 时，矩阵固定占用 $|V|^2$ 存储空间；<br>整体空间复杂度：$O(|V|^2)$ |
|  | 不适合稀疏图 | 现实业务大多是**稀疏图（边少点多）**；<br>矩阵里大量格子为空，存在巨额存储冗余，资源利用率极低 |

> 因此稠密图（Dense Graph）更适合用它存

#### 邻接表（Adjacency List）
这才是实际开发和算法里更常见的表示方式。
图里每个顶点，都维护一个“和自己直接相连的顶点列表”。也就是说：
- 图对象负责管理所有顶点
- 每个顶点自己记录“我能连到谁”

邻接表更适合稀疏图，因为它只存“真实存在的边”。空间复杂度大致是 $O(|V|+|E|)$ < $O(|V|^2)$ ，对稀疏图节省很多。

| 分类 | 核心要点 | 详细说明 |
| :--- | :--- | :--- |
| **优点** | 极度节省存储空间 | 专为**稀疏图**设计，只存储真实存在的边；<br>无冗余空数据，空间利用率极高 |
|  | 快速查询所有相邻顶点 | 获取某个顶点的全部邻居节点，直接读取该顶点的邻接列表即可，遍历高效便捷 |
| **缺点** | 判断两点连通效率偏低 | 想要确认顶点 `v` 与 `w` 是否直接相连；<br>需要遍历 `v` 的邻居列表逐个比对，无法做到 $O(1)$ 瞬时查询 |



In [ ]:
class Vertex:
    def __init__(self, key):
        self.id = key           # 表示我是谁，即顶点的编号/名字
        self.connectedTo = {}   # 字典，方便存储与我相连的顶点和权重

    def addNeighbor(self, nbr, weight=0):
        self.connectedTo[nbr] = weight  # 从当前顶点出发的边

    def __str__(self):
        return str(self.id) + ' connectedTo: ' + str([x.id for x in self.connectedTo])

    def getConnections(self):
        return self.connectedTo.keys()  # 返回与我相连的所有顶点

    def getId(self):
        return self.id

    def getWeight(self, nbr):
        return self.connectedTo[nbr]

In [ ]:
class Graph:
    def __init__(self):
        self.vertList = {}      # 字典，存储图中的所有顶点
        self.numVertices = 0    # 顶点的数量

    def addVertex(self, key):
        self.numVertices = self.numVertices + 1
        newVertex = Vertex(key)  # 创建一个新的顶点对象
        self.vertList[key] = newVertex  # 将新顶点添加到图中
        return newVertex
    
    def getVertex(self, n):
        if n in self.vertList:
            return self.vertList[n]
        else:
            return None
        
    def __contains__(self, n):      # 经典魔法方法，支持in操作符
        return n in self.vertList
    
    def addEdge(self, fromV, toV, weight=0):
        if fromV not in self.vertList:
            newFromV = self.addVertex(fromV)  # 如果起点不存在，先创建它
        if toV not in self.vertList:
            newToV = self.addVertex(toV)      # 如果终点不存在，先创建它
        self.vertList[fromV].addNeighbor(self.vertList[toV], weight)
        # 如果想实现无向图，下面这行代码也要加上使其双向
        # self.vertList[toV].addNeighbor(self.vertList[fromV], weight)

    def getVertices(self):
        return self.vertList.keys()  # 返回图中所有顶点的编号/名字
    
    def __iter__(self):
        return iter(self.vertList.values())  # 迭代器，返回图中所有顶点对象，可以写`for v in g:`来遍历图中的所有顶点